In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv("../data/train.csv")

In [3]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
df.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

MISSING VALUE

In [5]:
missing_values=df.isnull().sum()
missing_values[missing_values>0].sort_values(ascending=False)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

In [6]:
missing_percentage=(df.isnull().sum()/len(df))*100
missing_percentage[missing_percentage>0].sort_values(ascending=False)

PoolQC          99.520548
MiscFeature     96.301370
Alley           93.767123
Fence           80.753425
MasVnrType      59.726027
FireplaceQu     47.260274
LotFrontage     17.739726
GarageType       5.547945
GarageYrBlt      5.547945
GarageFinish     5.547945
GarageQual       5.547945
GarageCond       5.547945
BsmtExposure     2.602740
BsmtFinType2     2.602740
BsmtQual         2.534247
BsmtCond         2.534247
BsmtFinType1     2.534247
MasVnrArea       0.547945
Electrical       0.068493
dtype: float64

In [5]:
df["PoolQC"].value_counts(dropna=False)

PoolQC
NaN    1453
Gd        3
Ex        2
Fa        2
Name: count, dtype: int64

In [3]:
missing_columns=df.columns[df.isnull().any()].tolist()
print(missing_columns)

['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']


HANDLING FEATURES THAT DOSEN'T EXIST

In [8]:
none_cols=[
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu"
]
df[none_cols]=df[none_cols].fillna("None")

HANDLING BASEMENT FEATURES

In [11]:
bsmt_cat_cols=[
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2"
]
df[bsmt_cat_cols]=df[bsmt_cat_cols].fillna("None")

HANDLING GARAGE FEATURES

In [13]:
garage_cat_cols=[
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond"
]
df[garage_cat_cols]=df[garage_cat_cols].fillna("None")

ANALYZING & HANDLE REMAINING MISSING VALUES

In [14]:
df[[
    "LotFrontage",
    "MasVnrType",
    "MasVnrArea",
    "Electrical",
    "GarageYrBlt"
]].isnull().sum()

LotFrontage    259
MasVnrType     872
MasVnrArea       8
Electrical       1
GarageYrBlt     81
dtype: int64

In [15]:
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [16]:
df.isnull().sum().sum()

np.int64(0)

ENCODE CATEGORICAL FEATURES

In [17]:
categorical_cols=df.select_dtypes(include="object").columns
print(categorical_cols)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')


C:\Users\nithi\AppData\Local\Temp\ipykernel_25712\1181112195.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols=df.select_dtypes(include="object").columns


In [22]:
df=pd.get_dummies(df,columns=categorical_cols,drop_first=True)


KeyError: "None of [Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',\n       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',\n       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',\n       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',\n       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',\n       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',\n       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',\n       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',\n       'SaleType', 'SaleCondition'],\n      dtype='str')] are in the [columns]"

In [ ]:
df.select_dtypes(include="object").columns


(1460, 261)

FEATURES AND TARGET

In [25]:
x=df.drop("SalePrice",axis=1)
y=df["SalePrice"]

SPLIT THE DATASET

In [26]:
X_train,X_test,Y_train,Y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)
print(X_train.shape)
print(X_test.shape)

(1168, 260)
(292, 260)


SAVE THE PREPROCESSED DATASET

In [27]:
df.to_csv("../data/preprocessed_housing.csv",index=False)